# Phase 3: Data Cleaning, Preprocessing & The Extreme-Day Policy
**Financial Market Analytics & Investment Decision Support System for Indian Equities**
*CRISP-DM Phase: Data Preparation*

---

### Educational & Analytical Context
A foundational flaw in traditional data science preprocessing of financial time series is treating fat-tailed market shocks as "outliers" to be scrubbed. 
In equity risk analytics, **market crashes are not measurement errors—they are the risk**.

This notebook documents the validation, deduplication, trading calendar alignment, and forward-filling rules implemented in `src/clean/clean_market_data.py`. It presents an empirical **counterfactual demonstration** of the legacy $1.5 \times \text{IQR}$ filter versus our crash-preserving **Extreme-Day Policy (AGENTS.md Hard Rule 1)**.


## 1. Data Cleaning Pipeline Architecture (PRD §7.1)

Per **AGENTS.md Hard Rules 1 and 3**:
- **Raw Data Immutability:** Raw CSV files in `data/raw/` remain untouched.
- **Physical Error Dropping:** Only confirmed physical data errors are eliminated:
  - Non-positive prices ($Open, High, Low, Close \le 0$).
  - Inverted bars ($High < Low$).
  - Close violations ($Close < Low$ or $Close > High$).
  - Exact duplicates on `(ticker, date)`—the last observation is retained.
- **Extreme Day Policy:** Genuine market moves are **never** deleted. Any bar with $|z| > 4.0$ or $|\text{return}| \ge 10\%$ is flagged with `is_extreme = True` and preserved.
- **Forward-Filling:** Gaps of 1 to 2 consecutive trading days against the official NSE holiday calendar are forward-filled and explicitly tagged with `is_filled = True`. Gaps longer than 2 days are logged for investigation.


In [1]:
import sys
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook_connected"

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

con = duckdb.connect(str(project_root / "data" / "market.duckdb"), read_only=True)

df_summary = con.execute('''
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ticker) AS total_series,
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    SUM(CASE WHEN is_extreme THEN 1 ELSE 0 END) AS extreme_days_count,
    SUM(CASE WHEN is_filled THEN 1 ELSE 0 END) AS forward_filled_count
FROM prices
''').fetchdf()

df_summary


,total_rows,total_series,min_date,max_date,extreme_days_count,forward_filled_count
0,222947,84,2016-01-01,2026-10-19,1122.0,508.0


## 2. Calendar Alignment & Forward-Filling Policy (508 Gaps)

When a stock does not trade for 1 or 2 days due to temporary exchange technicalities or illiquidity, price series develop discontinuous gaps. 
To ensure continuous price paths for cumulative charting, the pipeline forward-fills these gaps with the previous close and zero volume, tagging them with `is_filled = True`.

> **Critical Statistical Governance:**  
> All subsequent statistical modules (volatility, return distribution, volume correlation, and portfolio covariance) must query `WHERE is_filled = FALSE` to prevent artificial zero-return clustering from deflating standard deviation estimates.


In [2]:
# Inspect forward-filled records across the 80 universe stocks
df_filled = con.execute('''
SELECT 
    u.cap_class,
    p.ticker,
    COUNT(*) AS filled_days,
    MIN(p.date) AS first_fill_date,
    MAX(p.date) AS last_fill_date
FROM prices p
JOIN universe u ON p.ticker = u.ticker
WHERE p.is_filled = TRUE
GROUP BY u.cap_class, p.ticker
ORDER BY filled_days DESC
''').fetchdf()

print(f"Total Forward-Filled Rows across Universe: {df_filled['filled_days'].sum()} rows across {len(df_filled)} stocks")
print("\nTop 10 Stocks with Forward-Filled Gaps:")
df_filled.head(10)


Total Forward-Filled Rows across Universe: 485 rows across 80 stocks

Top 10 Stocks with Forward-Filled Gaps:


,cap_class,ticker,filled_days,first_fill_date,last_fill_date
0,mid,AUROPHARMA.NS,7,2019-02-13,2026-06-26
1,mid,THERMAX.NS,7,2019-02-13,2026-06-26
2,mid,BHARATFORG.NS,7,2019-02-13,2026-06-26
3,small,GRANULES.NS,7,2019-02-13,2026-06-26
4,mid,ALKEM.NS,7,2019-02-13,2026-06-26
5,mid,MARICO.NS,6,2019-02-13,2026-06-26
6,mid,GLENMARK.NS,6,2019-02-13,2026-06-26
7,small,TATACHEM.NS,6,2019-02-13,2026-06-26
8,small,BATAINDIA.NS,6,2019-02-13,2026-06-26
9,large,BPCL.NS,6,2019-02-13,2026-06-26


## 3. Empirical Counterfactual: Legacy $1.5 \times \text{IQR}$ Outlier Deletion vs Extreme-Day Policy (PRD §7.2)

In standard generic data preprocessing, analysts frequently apply John Tukey's boxplot rule to delete "outliers":
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

Below, we empirically execute this legacy filter on our exact 10-year clean dataset across all 80 universe stocks to demonstrate the severe structural damage caused by deleting genuine crash days:


In [3]:
# Load clean returns excluding forward-filled rows
df_stocks = con.execute('''
SELECT p.ticker, p.date, p.close, p.simple_return, u.cap_class, u.sector
FROM prices p
JOIN universe u ON p.ticker = u.ticker
WHERE p.is_filled = FALSE AND p.simple_return IS NOT NULL
ORDER BY p.ticker, p.date
''').fetchdf()

cf_results = []
for ticker, group in df_stocks.groupby('ticker'):
    rets = group['simple_return']
    q1 = rets.quantile(0.25)
    q3 = rets.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outliers_mask = (rets < lower) | (rets > upper)
    wiped_count = int(outliers_mask.sum())
    wiped_pct = float(wiped_count / len(rets))
    
    # 1. Real metrics (Preserved crash days)
    real_vol = float(rets.std() * np.sqrt(252))
    cum_ret = (1 + rets).cumprod()
    peak = cum_ret.cummax()
    real_mdd = float(((cum_ret - peak) / peak).min())
    real_var95 = float(np.percentile(rets, 5))
    
    # 2. Counterfactual metrics (Wiped crash days)
    cf_rets = rets[~outliers_mask]
    cf_vol = float(cf_rets.std() * np.sqrt(252))
    cf_cum = (1 + cf_rets).cumprod()
    cf_peak = cf_cum.cummax()
    cf_mdd = float(((cf_cum - cf_peak) / cf_peak).min())
    cf_var95 = float(np.percentile(cf_rets, 5))
    
    cf_results.append({
        'ticker': ticker,
        'cap_class': group['cap_class'].iloc[0],
        'sector': group['sector'].iloc[0],
        'total_days': len(rets),
        'wiped_days': wiped_count,
        'wiped_pct': wiped_pct,
        'real_vol': real_vol,
        'cf_vol': cf_vol,
        'vol_understatement_pct': (real_vol - cf_vol) / real_vol * 100,
        'real_mdd': real_mdd,
        'cf_mdd': cf_mdd,
        'real_var95': real_var95,
        'cf_var95': cf_var95
    })

df_cf = pd.DataFrame(cf_results)


In [4]:
# Summary Aggregates: Real vs Counterfactual
total_wiped = df_cf['wiped_days'].sum()
avg_wiped_per_stock = df_cf['wiped_days'].mean()
avg_wiped_pct = df_cf['wiped_pct'].mean() * 100

summary_comparison = pd.DataFrame([
    {
        "Metric": "Total Market Days Removed",
        "Our Policy (Extreme Preserved)": "0 days (100% retained)",
        "Legacy 1.5xIQR Filter": f"{total_wiped:,} days ({avg_wiped_pct:.1f}% of data)"
    },
    {
        "Metric": "Average Annualized Volatility",
        "Our Policy (Extreme Preserved)": f"{df_cf['real_vol'].mean():.2%}",
        "Legacy 1.5xIQR Filter": f"{df_cf['cf_vol'].mean():.2%} (understated by {df_cf['vol_understatement_pct'].mean():.1f}%)"
    },
    {
        "Metric": "Average Maximum Drawdown",
        "Our Policy (Extreme Preserved)": f"{df_cf['real_mdd'].mean():.2%}",
        "Legacy 1.5xIQR Filter": f"{df_cf['cf_mdd'].mean():.2%}"
    },
    {
        "Metric": "Average Daily 95% Value at Risk (VaR)",
        "Our Policy (Extreme Preserved)": f"{df_cf['real_var95'].mean():.2%}",
        "Legacy 1.5xIQR Filter": f"{df_cf['cf_var95'].mean():.2%}"
    }
])

print("=== RISK METRIC DISTORTION: REAL VS COUNTERFACTUAL IQR ===")
summary_comparison


=== RISK METRIC DISTORTION: REAL VS COUNTERFACTUAL IQR ===


,Metric,Our Policy (Extreme Preserved),Legacy 1.5xIQR Filter
0,Total Market Days Removed,0 days (100% retained),"9,951 days (4.7% of data)"
1,Average Annualized Volatility,32.28%,24.46% (understated by 24.1%)
2,Average Maximum Drawdown,-57.66%,-53.32%
3,Average Daily 95% Value at Risk (VaR),-2.90%,-2.55%


## 4. Case Study: The COVID-19 Crash (March 2020) in Nifty 50

To visualize the specific days deleted by $1.5 \times \text{IQR}$, we inspect every trading day of **March 2020** for the Nifty 50 (`^NSEI`).
Notice how the single worst trading day in modern Indian market history—**23 March 2020 (-12.98%)**—is mechanically wiped out by the IQR filter, but faithfully preserved by our Extreme-Day Policy with `is_extreme = True`:


In [5]:
# Query Nifty March 2020 trading days
df_nifty_all = con.execute('''
SELECT date, close, simple_return, z_score, is_extreme
FROM prices
WHERE ticker = '^NSEI' AND is_filled = FALSE
ORDER BY date
''').fetchdf().dropna()

# Compute Nifty IQR thresholds
q1_nifty = df_nifty_all['simple_return'].quantile(0.25)
q3_nifty = df_nifty_all['simple_return'].quantile(0.75)
iqr_nifty = q3_nifty - q1_nifty
lower_nifty = q1_nifty - 1.5 * iqr_nifty
upper_nifty = q3_nifty + 1.5 * iqr_nifty

df_mar2020 = df_nifty_all[df_nifty_all['date'].str.startswith('2020-03')].copy()
df_mar2020['iqr_deleted'] = (df_mar2020['simple_return'] < lower_nifty) | (df_mar2020['simple_return'] > upper_nifty)
df_mar2020['simple_return_pct'] = (df_mar2020['simple_return'] * 100).round(2)
df_mar2020['close'] = df_mar2020['close'].round(2)
df_mar2020['z_score'] = df_mar2020['z_score'].round(2)

print(f"Nifty 50 10-Year IQR Cutoffs: Lower = {lower_nifty*100:.2f}%, Upper = {upper_nifty*100:.2f}%")
print(f"March 2020 Days Wiped by IQR: {df_mar2020['iqr_deleted'].sum()} out of {len(df_mar2020)} trading days")

display_cols = ['date', 'close', 'simple_return_pct', 'z_score', 'is_extreme', 'iqr_deleted']
df_mar2020[display_cols]


Nifty 50 10-Year IQR Cutoffs: Lower = -1.94%, Upper = 2.08%
March 2020 Days Wiped by IQR: 15 out of 21 trading days


,date,close,simple_return_pct,z_score,is_extreme,iqr_deleted
1017,2020-03-02,11132.75,-0.62,-0.65,False,False
1018,2020-03-03,11303.30,1.53,1.46,False,False
1019,2020-03-04,11251.00,-0.46,-0.50,False,False
1020,2020-03-05,11269.00,0.16,0.11,False,False
1021,2020-03-06,10989.45,-2.48,-2.49,False,True
1022,2020-03-09,10451.45,-4.90,-4.86,True,True
1023,2020-03-11,10458.40,0.07,0.02,False,False
1024,2020-03-12,9590.15,-8.30,-8.22,True,True
1025,2020-03-13,9955.20,3.81,3.70,False,True
1026,2020-03-16,9197.40,-7.61,-7.54,True,True


In [6]:
# Plotly Visualisation of Nifty Return Distribution: Real vs IQR-Truncated
fig_dist = go.Figure()

fig_dist.add_trace(go.Histogram(
    x=df_nifty_all['simple_return'] * 100,
    name='Real Distribution (Preserved Tails)',
    nbinsx=100,
    opacity=0.6,
    marker_color='#1f77b4'
))

nifty_retained = df_nifty_all[~((df_nifty_all['simple_return'] < lower_nifty) | (df_nifty_all['simple_return'] > upper_nifty))]
fig_dist.add_trace(go.Histogram(
    x=nifty_retained['simple_return'] * 100,
    name='Counterfactual 1.5xIQR Filtered',
    nbinsx=100,
    opacity=0.6,
    marker_color='#ff7f0e'
))

fig_dist.add_vline(x=lower_nifty*100, line_dash="dash", line_color="red", annotation_text="IQR Lower (-1.94%)")
fig_dist.add_vline(x=upper_nifty*100, line_dash="dash", line_color="green", annotation_text="IQR Upper (+2.08%)")

# Add highlight for 23 March 2020
m23_ret = df_mar2020.loc[df_mar2020['date'] == '2020-03-23', 'simple_return_pct'].values[0]
fig_dist.add_annotation(
    x=m23_ret,
    y=10,
    text=f"23 Mar 2020: {m23_ret}%<br>(Deleted by IQR!)",
    showarrow=True,
    arrowhead=1,
    arrowcolor="red",
    ax=0,
    ay=-40
)

fig_dist.update_layout(
    title="Nifty 50 Daily Return Distribution: The Danger of Outlier Deletion",
    xaxis_title="Daily Simple Return (%)",
    yaxis_title="Trading Days Count",
    barmode='overlay',
    width=900,
    height=480
)
fig_dist.show()


In [7]:
# Compute exact summary statistics for the markdown narrative
wiped_total = int(df_cf['wiped_days'].sum())
avg_wiped = float(df_cf['wiped_days'].mean())
avg_understate = float(df_cf['vol_understatement_pct'].mean())
mar_wiped_cnt = int(df_mar2020['iqr_deleted'].sum())
mar_total_cnt = len(df_mar2020)
m23_ret = df_mar2020.loc[df_mar2020['date'] == '2020-03-23', 'simple_return_pct'].values[0]

from IPython.display import Markdown

interpretation_text = (
    f'> **Analytical Interpretation & Methodological Lesson:**\n'
    f'> Applying a standard 1.5x IQR outlier filter to the 80 universe stocks would have mechanically '
    f'erased **{wiped_total:,} genuine market trading days** (averaging **{avg_wiped:.1f} days per stock**), '
    f'completely distorting risk measurement.\n'
    f'> As demonstrated in the summary metrics, deleting these tail observations artificially suppresses annualized '
    f'equity volatility from **{df_cf["real_vol"].mean():.1%} down to {df_cf["cf_vol"].mean():.1%}** '
    f'— an understatement of **{avg_understate:.1f}%** — while creating a false sense of security regarding maximum '
    f'drawdown and 95% Value at Risk.\n'
    f'> In March 2020 alone, the IQR filter erased **{mar_wiped_cnt} of {mar_total_cnt} trading days** in the Nifty 50, '
    f'including the catastrophic **23 March 2020 drop of {m23_ret}%**. Our **Extreme-Day Policy** correctly preserves '
    f'all genuine macroeconomic moves with an `is_extreme` flag, ensuring investors receive honest, uncompromised risk assessments.'
)
Markdown(interpretation_text)
con.close()


## 5. Summary & Transition to Exploratory Data Analysis (Phase 4)

### Final Conclusions
1. **Physical Errors vs Genuine Shocks:** The pipeline strictly decouples data transmission artifacts (dropped) from genuine market volatility (preserved).
2. **Forward-Filling Integrity:** Exactly 508 calendar gaps are forward-filled for charting but quarantined with `is_filled = True` so statistical estimators remain unbiased.
3. **Statistical Validity:** Financial risk modeling must never truncate legitimate extreme returns.

We now transition to **Phase 4: Exploratory Data Analysis** (`notebooks/04_eda.ipynb`) to examine long-term sector growth, rolling volatility regimes, crisis drawdowns, and cross-asset correlation dynamics.
